In [30]:
import pandas as pd

df1 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
df2 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
df3 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')
df4 = pd.read_csv("../dataset/raw/play_off_box_scores_2010_2024.csv")

df = pd.concat([df1, df2, df3, df4], axis = 0)

df = df.drop(columns = ['jerseyNum', 'comment'], axis = 1)
df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())
df = df.dropna()

# Concertendo minutos para float

def convert_minutes(min_str):
    if isinstance(min_str, str):
        mins, secs = map(int, min_str.split(':'))
        return mins + secs/60
    else:
        return min_str  # Se já for número, deixa quieto

df['minutes'] = df['minutes'].apply(convert_minutes)

df.columns

Index(['season_year', 'game_date', 'gameId', 'matchup', 'teamId', 'teamCity',
       'teamName', 'teamTricode', 'teamSlug', 'personId', 'personName',
       'position', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted',
       'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted',
       'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted',
       'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive',
       'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers',
       'foulsPersonal', 'points', 'plusMinusPoints'],
      dtype='object')

In [29]:
player_stats = df.drop(columns = [ 'game_date', 'gameId', 'matchup', 'teamId', 'teamCity', 'teamTricode', 'teamSlug', 'personId'], axis = 1).copy()
player_stats = player_stats.groupby(by = ['personName', 'teamName', 'season_year', 'position']).mean(numeric_only=True).reset_index()
player_stats['fieldGoalsPercentage'] = player_stats.apply(lambda x: (x['fieldGoalsMade'] / x['fieldGoalsAttempted']) * 100 if x['fieldGoalsAttempted'] != 0 else 0, axis = 1)
player_stats['threePointersPercentage'] = player_stats.apply(lambda x: (x['threePointersMade'] / x['threePointersAttempted']) * 100 if x['threePointersAttempted'] != 0 else 0, axis = 1)
player_stats['freeThrowsPercentage'] = player_stats.apply(lambda x: (x['freeThrowsMade'] / x['freeThrowsAttempted']) * 100 if x['freeThrowsAttempted'] != 0 else 0, axis = 1)
player_stats.to_csv('../dataset/clean/stats_by_player.csv', index = False)

player_stats

,personName,teamName,season_year,position,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
0,AJ Green,Bucks,2022-23,G,9.855714,1.514286,3.571429,42.400000,1.257143,3.000000,...,0.171429,1.114286,1.285714,0.628571,0.171429,0.000000,0.257143,0.885714,4.400000,-0.742857
1,AJ Green,Bucks,2023-24,G,10.969345,1.482143,3.500000,42.346939,1.232143,3.017857,...,0.160714,0.982143,1.142857,0.535714,0.160714,0.071429,0.214286,0.875000,4.500000,0.892857
2,AJ Griffin,Hawks,2022-23,F,19.083660,3.392157,7.313725,46.380697,1.470588,3.745098,...,0.470588,1.607843,2.078431,0.980392,0.588235,0.176471,0.509804,1.176471,8.666667,1.058824
3,AJ Griffin,Hawks,2022-23,G,20.389683,3.571429,7.619048,46.875000,1.238095,3.238095,...,0.619048,1.619048,2.238095,1.095238,0.571429,0.142857,0.761905,1.285714,9.380952,0.428571
4,AJ Griffin,Hawks,2023-24,F,8.998889,0.933333,2.866667,32.558140,0.466667,1.666667,...,0.066667,0.933333,1.000000,0.333333,0.066667,0.133333,0.400000,0.333333,2.466667,-2.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9931,Zion Williamson,Pelicans,2022-23,F,32.968966,9.827586,16.172414,60.767591,0.241379,0.655172,...,2.000000,4.965517,6.965517,4.586207,1.103448,0.551724,3.413793,2.241379,26.000000,5.137931
9932,Zion Williamson,Pelicans,2023-24,F,31.527619,8.914286,15.628571,57.038391,0.085714,0.257143,...,1.742857,4.057143,5.800000,5.028571,1.085714,0.671429,2.757143,2.271429,22.871429,2.042857
9933,Zoran Dragic,Heat,2014-15,G,6.178333,0.900000,2.200000,40.909091,0.300000,0.900000,...,0.300000,0.200000,0.500000,0.400000,0.200000,0.000000,0.500000,0.500000,2.200000,-1.300000
9934,Zoran Dragic,Suns,2014-15,G,2.238889,0.333333,1.333333,25.000000,0.000000,0.833333,...,0.333333,0.166667,0.500000,0.166667,0.000000,0.000000,0.000000,0.166667,1.000000,-0.333333
